# 01 — Data Collection and SQLite Database Setup
**Project:** Charlotte NPA Housing Affordability Analysis  
**Data Source:** Mecklenburg County Quality of Life Explorer  
**Target Variable:** Home Sales Price (regression) / Displacement Risk (classification)  

This notebook loads all raw CSVs into individual tables in a SQLite database, then merges them into a single analysis-ready table keyed by NPA ID. Year columns are tracked for every feature to document temporal alignment. All features use 2023 data except Voter Participation which uses 2025.

In [19]:
import sqlite3
import pandas as pd
import os

# Dynamically finds the repo root regardless of who is running it or where they cloned it
base = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f'Base path set to: {base}')



# Connect to the database (creates it if it does not exist)
conn = sqlite3.connect(os.path.join(base, 'data', 'charlotte_housing.db'))
cursor = conn.cursor()

print('Database connected successfully.')
print(f'Database location: {os.path.join(base, "data", "charlotte_housing.db")}')

Base path set to: c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis
Database connected successfully.
Database location: c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\data\charlotte_housing.db


In [2]:
# Home Sales Price — TARGET VARIABLE
# Source: Quality of Life Explorer
# Description: Median home sales price per NPA
# Year: 2023
# Note: Used as the target variable for both models.
#       For regression it is kept as a continuous dollar value.
#       For classification it is converted to a Displacement Risk label
#       using the 30 percent affordability rule in the EDA notebook.

hv = pd.read_csv(os.path.join(base, 'data', 'raw', 'Home_Sales_Price.csv'))
hv = hv[['NPA', '2023']].copy()
hv.columns = ['npa_id', 'home_sales_price']
hv['home_sales_price'] = hv['home_sales_price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
hv['home_sales_price'] = hv['home_sales_price'].replace('--', float('nan'))
hv['home_sales_price'] = pd.to_numeric(hv['home_sales_price'], errors='coerce')
hv['data_year'] = 2023
hv.to_sql('home_sales_price', conn, if_exists='replace', index=False)
print(f'home_sales_price loaded: {len(hv)} rows, {hv["home_sales_price"].isna().sum()} nulls')

home_sales_price loaded: 459 rows, 15 nulls


In [3]:
# Tree Canopy
# Source: https://maps.mecklenburgcountync.gov/qol/#3/
# Description: Percentage of land area covered by tree canopy
# Year: 2023

tc = pd.read_csv(os.path.join(base, 'data', 'raw', 'Tree_Canopy.csv'))
tc = tc[['NPA', '2023']].copy()
tc.columns = ['npa_id', 'tree_canopy_pct']
tc['tree_canopy_pct'] = tc['tree_canopy_pct'].astype(str).str.replace('%', '', regex=False).str.strip()
tc['tree_canopy_pct'] = pd.to_numeric(tc['tree_canopy_pct'], errors='coerce')
tc['data_year'] = 2023
tc.to_sql('tree_canopy', conn, if_exists='replace', index=False)
print(f'tree_canopy loaded: {len(tc)} rows, {tc["tree_canopy_pct"].isna().sum()} nulls')

tree_canopy loaded: 459 rows, 0 nulls


In [4]:
# Education Level
# Source: https://maps.mecklenburgcountync.gov/qol/#20/
# Description: Percentage of adults over age 25 with a Bachelor's degree or higher
# Year: 2023

ed = pd.read_csv(os.path.join(base, 'data', 'raw', 'Education_Level_-_Bachelor_s_Degree.csv'))
ed = ed[['NPA', '2023']].copy()
ed.columns = ['npa_id', 'bachelors_pct']
ed['bachelors_pct'] = ed['bachelors_pct'].astype(str).str.replace('%', '', regex=False).str.strip()
ed['bachelors_pct'] = pd.to_numeric(ed['bachelors_pct'], errors='coerce')
ed['data_year'] = 2023
ed.to_sql('education', conn, if_exists='replace', index=False)
print(f'education loaded: {len(ed)} rows, {ed["bachelors_pct"].isna().sum()} nulls')

education loaded: 459 rows, 2 nulls


In [5]:
# Water Consumption
# Source: https://maps.mecklenburgcountync.gov/qol/#27/
# Description: Average daily water consumption for single-family housing units (gallons per day per unit)
# Year: 2023

wc = pd.read_csv(os.path.join(base, 'data', 'raw', 'Water_Consumption.csv'))
wc = wc[['NPA', '2023']].copy()
wc.columns = ['npa_id', 'water_consumption_gpd']
wc['water_consumption_gpd'] = pd.to_numeric(wc['water_consumption_gpd'], errors='coerce')
wc['data_year'] = 2023
wc.to_sql('water_consumption', conn, if_exists='replace', index=False)
print(f'water_consumption loaded: {len(wc)} rows, {wc["water_consumption_gpd"].isna().sum()} nulls')

water_consumption loaded: 459 rows, 24 nulls


In [6]:
# Rental Costs
# Source: Quality of Life Explorer
# Description: Median monthly rental cost per NPA
# Year: 2023
# Note: Replaces Public Health Insurance which was only available through 2017

rc = pd.read_csv(os.path.join(base, 'data', 'raw', 'Rental_Costs.csv'))
rc = rc[['NPA', '2023']].copy()
rc.columns = ['npa_id', 'median_rent']
rc['median_rent'] = rc['median_rent'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
rc['median_rent'] = rc['median_rent'].replace('--', float('nan'))
rc['median_rent'] = pd.to_numeric(rc['median_rent'], errors='coerce')
rc['data_year'] = 2023
rc.to_sql('rental_costs', conn, if_exists='replace', index=False)
print(f'rental_costs loaded: {len(rc)} rows, {rc["median_rent"].isna().sum()} nulls')

rental_costs loaded: 459 rows, 52 nulls


In [7]:
# Student Absenteeism
# Source: https://maps.mecklenburgcountync.gov/qol/#66/
# Description: Percentage of CMS students absent 10 percent or more of school days
# Year: 2023

ab = pd.read_csv(os.path.join(base, 'data', 'raw', 'Student_Absenteeism.csv'))
ab = ab[['NPA', '2023']].copy()
ab.columns = ['npa_id', 'absenteeism_pct']
ab['absenteeism_pct'] = ab['absenteeism_pct'].astype(str).str.replace('%', '', regex=False).str.strip()
ab['absenteeism_pct'] = pd.to_numeric(ab['absenteeism_pct'], errors='coerce')
ab['data_year'] = 2023
ab.to_sql('absenteeism', conn, if_exists='replace', index=False)
print(f'absenteeism loaded: {len(ab)} rows, {ab["absenteeism_pct"].isna().sum()} nulls')

absenteeism loaded: 459 rows, 3 nulls


In [8]:
# Household Income
# Source: https://maps.mecklenburgcountync.gov/qol/#37/
# Description: Median household income
# Year: 2023

inc = pd.read_csv(os.path.join(base, 'data', 'raw', 'Household_Income.csv'))
inc = inc[['NPA', '2023']].copy()
inc.columns = ['npa_id', 'household_income']
inc['household_income'] = inc['household_income'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
inc['household_income'] = pd.to_numeric(inc['household_income'], errors='coerce')
inc['data_year'] = 2023
inc.to_sql('household_income', conn, if_exists='replace', index=False)
print(f'household_income loaded: {len(inc)} rows, {inc["household_income"].isna().sum()} nulls')

household_income loaded: 459 rows, 14 nulls


In [9]:
# Housing Size
# Source: https://maps.mecklenburgcountync.gov/qol/#6/
# Description: Average heated living space of single-family housing units (avg ft2)
# Year: 2023

hs = pd.read_csv(os.path.join(base, 'data', 'raw', 'Housing_Size.csv'))
hs = hs[['NPA', '2023']].copy()
hs.columns = ['npa_id', 'housing_size_sqft']
hs['housing_size_sqft'] = hs['housing_size_sqft'].astype(str).str.replace(',', '', regex=False).str.strip()
hs['housing_size_sqft'] = pd.to_numeric(hs['housing_size_sqft'], errors='coerce')
hs['data_year'] = 2023
hs.to_sql('housing_size', conn, if_exists='replace', index=False)
print(f'housing_size loaded: {len(hs)} rows, {hs["housing_size_sqft"].isna().sum()} nulls')

housing_size loaded: 459 rows, 22 nulls


In [10]:
# Voter Participation
# Source: https://maps.mecklenburgcountync.gov/qol/#48/
# Description: Percentage of registered voters that voted in the general election
# Year: 2025 (most recent general election available)
# NOTE: All other features use 2023. This two year gap is documented
#       in the year columns and addressed in the report limitations section.

vp = pd.read_csv(os.path.join(base, 'data', 'raw', 'Voter_Participation.csv'))
vp = vp[['NPA', '2025']].copy()
vp.columns = ['npa_id', 'voter_participation_pct']
vp['voter_participation_pct'] = vp['voter_participation_pct'].astype(str).str.replace('%', '', regex=False).str.strip()
vp['voter_participation_pct'] = pd.to_numeric(vp['voter_participation_pct'], errors='coerce')
vp['data_year'] = 2025
vp.to_sql('voter_participation', conn, if_exists='replace', index=False)
print(f'voter_participation loaded: {len(vp)} rows, {vp["voter_participation_pct"].isna().sum()} nulls')
print('NOTE: Voter Participation uses 2025 data. All other features use 2023.')

voter_participation loaded: 459 rows, 0 nulls
NOTE: Voter Participation uses 2025 data. All other features use 2023.


In [11]:
# Verify all tables are in the database
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print('Tables in charlotte_housing.db:')
print(tables.to_string(index=False))

Tables in charlotte_housing.db:
               name
       npa_features
   home_sales_price
        tree_canopy
          education
  water_consumption
       rental_costs
        absenteeism
   household_income
       housing_size
voter_participation


In [12]:
# Drop and recreate the merged table cleanly
cursor.execute('DROP TABLE IF EXISTS npa_features')
conn.commit()

merge_query = """
CREATE TABLE npa_features AS
SELECT
    hv.npa_id,

    -- Target variable
    hv.home_sales_price,

    -- Features
    tc.tree_canopy_pct,
    ed.bachelors_pct,
    wc.water_consumption_gpd,
    rc.median_rent,
    ab.absenteeism_pct,
    inc.household_income,
    hs.housing_size_sqft,
    vp.voter_participation_pct,

    -- Year documentation for each feature
    hv.data_year  AS home_sales_price_year,
    tc.data_year  AS tree_canopy_year,
    ed.data_year  AS education_year,
    wc.data_year  AS water_consumption_year,
    rc.data_year  AS rental_costs_year,
    ab.data_year  AS absenteeism_year,
    inc.data_year AS household_income_year,
    hs.data_year  AS housing_size_year,
    vp.data_year  AS voter_participation_year

FROM home_sales_price hv
LEFT JOIN tree_canopy        tc  ON hv.npa_id = tc.npa_id
LEFT JOIN education          ed  ON hv.npa_id = ed.npa_id
LEFT JOIN water_consumption  wc  ON hv.npa_id = wc.npa_id
LEFT JOIN rental_costs       rc  ON hv.npa_id = rc.npa_id
LEFT JOIN absenteeism        ab  ON hv.npa_id = ab.npa_id
LEFT JOIN household_income   inc ON hv.npa_id = inc.npa_id
LEFT JOIN housing_size       hs  ON hv.npa_id = hs.npa_id
LEFT JOIN voter_participation vp  ON hv.npa_id = vp.npa_id
"""

cursor.execute(merge_query)
conn.commit()
print('npa_features table created successfully.')

npa_features table created successfully.


In [13]:
# Confirm the final merged table
final_df = pd.read_sql('SELECT * FROM npa_features', conn)

print(f'Shape: {final_df.shape}')
print(f'\nFirst 5 rows:')
print(final_df.head().to_string())
print(f'\nMissing values per column:')
print(final_df.isnull().sum())

Shape: (459, 19)

First 5 rows:
   npa_id  home_sales_price  tree_canopy_pct  bachelors_pct  water_consumption_gpd  median_rent  absenteeism_pct  household_income  housing_size_sqft  voter_participation_pct  home_sales_price_year  tree_canopy_year  education_year  water_consumption_year  rental_costs_year  absenteeism_year  household_income_year  housing_size_year  voter_participation_year
0       2          488364.0             38.1           35.2                  125.0       1252.0             33.0           75084.0             1720.0                     31.6                   2023              2023            2023                    2023               2023              2023                   2023               2023                      2025
1       3          667092.0             23.1           85.4                  180.0       1883.0              6.6          117630.0             2807.0                     27.4                   2023              2023            2023               

In [14]:
# Show year alignment across all features
year_cols = [c for c in final_df.columns if c.endswith('_year')]
print('Data year by feature:')
print(final_df[year_cols].drop_duplicates().to_string(index=False))

Data year by feature:
 home_sales_price_year  tree_canopy_year  education_year  water_consumption_year  rental_costs_year  absenteeism_year  household_income_year  housing_size_year  voter_participation_year
                  2023              2023            2023                    2023               2023              2023                   2023               2023                      2025


In [15]:
# Close the connection
conn.close()
print('Connection closed. Database is ready.')
print(f'All subsequent notebooks should connect to:')
print(os.path.join(base, 'data', 'charlotte_housing.db'))

Connection closed. Database is ready.
All subsequent notebooks should connect to:
C:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\data\charlotte_housing.db
